# CRAVE fMRI Preprocessing and quality control

1. BIDS conversion (dcm2bids)
2. fMRIPrep 25.2.6
3. Motion QC: filtered framewise displacement (FD), censoring, run and participant exclusion
4. Multi-echo denoising with tedana, then warp to MNI space
5. QC summary tables and figure

Runs on the lab conda environment`tedana`. Steps 1, 2 and 4 take hours.

In [ ]:
import re
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt

ROOT = Path("/srv/lab/fmri/crave")
BIDS = ROOT / "crave_bids"
CODE = BIDS / "code"
FMRIPREP = ROOT / "derivatives" / "fmriprep"
TEDANA = ROOT / "derivatives" / "tedana"
QC = ROOT / "derivatives" / "qc"
SIF = ROOT / "containers" / "fmriprep-25.2.6.sif"   # fMRIPrep container (also has ANTs)

TR = 1.761                                          # seconds
ECHO_TIMES = [14.20, 38.92, 63.64, 88.36, 113.08]   # ms
(QC / "censor").mkdir(parents=True, exist_ok=True)

## 1. BIDS conversion

1. write the dcm2bids config (T1w; VS, MID and LB bold + SBRef for echoes 1–5; AP/PA spin-echo fieldmaps)
2. convert each participant with dcm2bids into `sub-<ID>/ses-01`
3. trim echoes so all five echoes of a run have the same number of volumes (originals backed up)
4. relabel the two sessions whose folders had the wrong ID

In [ ]:
import json
import glob
import os
import shutil
from collections import defaultdict
import nibabel as nib

RAW = ROOT / "raw_mri"                       # one study_*_<ID> folder per participant
CONFIG = CODE / "config.json"

In [ ]:
# 1. dcm2bids config
desc = [{"datatype": "anat", "suffix": "T1w", "criteria": {"SeriesDescription": "t1_mprage_0.9mm"}}]
bold_ids = []
for task in ["VS", "MID", "LB"]:
    for echo in range(1, 6):
        bold_id = f"func_{task}_echo{echo}"
        bold_ids.append(bold_id)
        desc.append({"id": bold_id, "datatype": "func", "suffix": "bold",
                     "custom_entities": f"task-{task}_echo-{echo}",
                     "criteria": {"SeriesDescription": f"{task}_epi_mc_2mm_task_run-0?", "EchoNumber": echo},
                     "sidecar_changes": {"TaskName": task}})
        desc.append({"datatype": "func", "suffix": "sbref",
                     "custom_entities": f"task-{task}_echo-{echo}",
                     "criteria": {"SeriesDescription": f"{task}_epi_mc_2mm_task_run-0?_SBRef", "EchoNumber": echo}})
for direction in ["AP", "PA"]:
    desc.append({"datatype": "fmap", "suffix": "epi", "custom_entities": f"dir-{direction}",
                 "criteria": {"SeriesDescription": f"cmrr_mbep2d_se_{direction}_rest"},
                 "sidecar_changes": {"IntendedFor": bold_ids}})

with open(CONFIG, "w") as f:
    json.dump({"descriptions": desc}, f, indent=2)
print("Wrote", len(desc), "rules to", CONFIG)

In [ ]:
# 2. convert every participant (skips participants that are already in BIDS)
for raw_dir in sorted(RAW.glob("study_*")):
    sub = raw_dir.name.split("_")[-1]
    if (BIDS / f"sub-{sub}").exists():
        continue
    subprocess.run(["dcm2bids", "-d", str(raw_dir), "-p", sub, "-s", "01", "-c", str(CONFIG), "-o", str(BIDS)])

In [ ]:
# 3. make all echoes of a run the same length (trim to the shortest; originals copied to bids_backup)
BACKUP = str(ROOT / "bids_backup")
runs = defaultdict(list)
for f in sorted(glob.glob(str(BIDS) + "/sub-*/ses-*/func/*_echo-*_bold.nii.gz")):
    runs[f.split("_echo-")[0]].append(f)

for run, files in runs.items():
    nvols = {f: nib.load(f).shape[3] for f in files}
    if len(set(nvols.values())) == 1:
        continue
    n = min(nvols.values())
    print("MISMATCH:", os.path.basename(run), list(nvols.values()), "-> trim to", n)
    for f, v in nvols.items():
        if v > n:
            backup = f.replace(str(BIDS), BACKUP)
            os.makedirs(os.path.dirname(backup), exist_ok=True)
            shutil.copy2(f, backup)
            nib.save(nib.load(backup).slicer[:, :, :, :n], f)

In [ ]:
# 4. two mislabelled sessions
#!python {CODE}/relabel_subject.py 08533 085333 --apply
#!python {CODE}/relabel_subject.py 2 633368 --apply

## 2. fMRIPrep

fMRIPrep 25.2.6 in Apptainer, one participant at a time (up to 2 attempts). Echo-wise outputs are kept for tedana;
outputs in MNI152NLin2009cAsym 2 mm, T1w space and CIFTI 91k. A `.done` file marks finished participants.

In [ ]:
LOGS = ROOT / "derivatives" / "logs"
SCRATCH = ROOT / "work"
LOGS.mkdir(parents=True, exist_ok=True)

subs = sorted(d.name[4:] for d in BIDS.glob("sub-*") if d.is_dir())
for sub in subs:
    done = LOGS / f"sub-{sub}.done"
    if done.exists():
        continue
    work = SCRATCH / f"sub-{sub}"
    work.mkdir(parents=True, exist_ok=True)
    cmd = ["apptainer", "run", "--cleanenv",
           "-B", f"{ROOT}:/data", "-B", f"{work}:/work", str(SIF),
           "/data/crave_bids", "/data/derivatives/fmriprep", "participant",
           "--participant-label", sub,
           "--fs-license-file", "/data/license.txt",
           "-w", "/work",
           "--me-output-echos",
           "--output-spaces", "MNI152NLin2009cAsym:res-2", "T1w", #Using MNI template
           "--cifti-output", "91k",
           "--nthreads", "8", "--omp-nthreads", "4", "--mem-mb", "32000"]
    for attempt in [1, 2]:
        with open(LOGS / f"sub-{sub}_attempt{attempt}.log", "w") as log:
            rc = subprocess.call(cmd, stdout=log, stderr=subprocess.STDOUT, cwd=str(ROOT))
        if rc == 0:
            break
    if rc == 0:
        done.touch()
        shutil.rmtree(work, ignore_errors=True)
        print("finished sub-" + sub)
    else:
        print("FAILED sub-" + sub + " - see", LOGS)

## 3. Motion QC

Breathing shows up in the motion estimates (mostly `trans_y`), so the six motion parameters are
low-pass filtered at 0.1 Hz before computing FD.

- censor a volume if filtered FD > 0.3 mm **or** raw FD > 0.9 mm
- exclude a run if < 50% of volumes are kept **or** it is shorter than 50% of the task's median length
- exclude a participant with fewer than 3 usable Vampire Survivors (VS) rounds

In [ ]:
MOTION = ["trans_x", "trans_y", "trans_z", "rot_x", "rot_y", "rot_z"]

def framewise_displacement(motion):
    # Power et al. (2012): rotations (radians) become mm on a 50 mm sphere
    d = np.abs(np.diff(motion, axis=0))
    d[:, 3:] = d[:, 3:] * 50
    return np.concatenate([[0], d.sum(axis=1)])

def lowpass(motion, cutoff=0.1):
    b, a = butter(2, cutoff / (0.5 / TR), btype="low")
    return filtfilt(b, a, motion, axis=0, padlen=min(9, len(motion) - 1)) 

In [ ]:
rows = []
for conf_file in sorted(FMRIPREP.glob("sub-*/ses-01/func/*_desc-confounds_timeseries.tsv")):
    name = conf_file.name
    sub = name.split("_")[0]
    run_label = name.split("task-")[1].split("_desc")[0]          # e.g. VS_run-01, LB
    task = run_label.split("_")[0]

    conf = pd.read_csv(conf_file, sep="\t")
    motion = conf[MOTION].fillna(0).to_numpy()
    fd_raw = framewise_displacement(motion)
    fd_filt = framewise_displacement(lowpass(motion))
    censored = (fd_filt > 0.3) | (fd_raw > 0.9)
    censored[0] = False
    # there are a few participants with high motion so this is one way to keep them
    # censor file used by the first-level models (1 = censored volume)
    censor_file = QC / "censor" / name.replace("desc-confounds", "desc-censor")
    pd.DataFrame({"censored": censored.astype(int), "fd_raw": fd_raw, "fd_filtered": fd_filt}).to_csv(censor_file, sep="\t", index=False)

    rows.append({"sub": sub, "task": task, "run_label": run_label, "n_vols": len(conf),
                 "mean_fd_raw": fd_raw[1:].mean(), "mean_fd_filt": fd_filt[1:].mean(),
                 "n_censored": censored.sum(), "pct_censored": 100 * censored.mean(),
                 "pct_retained": 100 - 100 * censored.mean(),
                 "minutes_retained": (len(conf) - censored.sum()) * TR / 60})

run_qc = pd.DataFrame(rows)
run_qc.head()

In [ ]:
# run exclusion: too much censoring or an bad/incomplete run
task_median = run_qc.groupby("task")["n_vols"].transform("median")
run_qc["incomplete"] = run_qc["n_vols"] < 0.5 * task_median
run_qc["run_excluded"] = (run_qc["pct_retained"] < 50) | run_qc["incomplete"]
run_qc.to_csv(QC / "run_qc.csv", index=False)

print("Excluded runs:")
run_qc[run_qc["run_excluded"]][["sub", "run_label", "n_vols", "pct_retained"]]

In [ ]:
# participant exclusion: fewer than 3 usable VS rounds
vs_kept = run_qc[(run_qc["task"] == "VS") & ~run_qc["run_excluded"]].groupby("sub").size()
participant_qc = pd.DataFrame({"vs_rounds_kept": vs_kept}).reindex(run_qc["sub"].unique()).fillna(0)
participant_qc["mean_fd_filt"] = run_qc.groupby("sub")["mean_fd_filt"].mean()
participant_qc["mean_pct_censored"] = run_qc.groupby("sub")["pct_censored"].mean()
participant_qc["participant_excluded"] = participant_qc["vs_rounds_kept"] < 3
participant_qc.to_csv(QC / "participant_qc.csv")

print("Excluded participants:", list(participant_qc.index[participant_qc["participant_excluded"]]))

In [ ]:
# figure: % censored per run (red outline = run excluded)
table = run_qc.pivot(index="sub", columns="run_label", values="pct_censored")
excluded = run_qc.pivot(index="sub", columns="run_label", values="run_excluded").fillna(0).astype(bool)
table = table.loc[table.mean(axis=1).sort_values(ascending=False).index]
excluded = excluded.loc[table.index, table.columns]

fig, ax = plt.subplots(figsize=(10, 12))
im = ax.imshow(table, cmap="Blues", vmin=0, vmax=50, aspect="auto")
for i in range(table.shape[0]):
    for j in range(table.shape[1]):
        if excluded.iloc[i, j]:
            ax.add_patch(plt.Rectangle((j - 0.45, i - 0.45), 0.9, 0.9, fill=False, edgecolor="red", lw=2))
ax.set_xticks(range(table.shape[1]))
ax.set_xticklabels(table.columns, rotation=45, ha="right")
ax.set_yticks(range(table.shape[0]))
ax.set_yticklabels(table.index)
fig.colorbar(im, label="% of volumes censored")
ax.set_title("Censored volumes per run (red = run excluded: < 50% kept or incomplete run)")
fig.tight_layout()
fig.savefig(QC / "fd_heatmap.png", dpi=150)

## 4. tedana (multi-echo ICA) and warp to MNI

For each complete run: tedana on the five fMRIPrep echoes (native space), then
`antsApplyTransforms` from the fMRIPrep container (boldref → T1w → MNI152NLin2009cAsym, 2 mm).
The VS runs use these denoised data. The loot-box (LB) run is too short for ICA, so it uses the fMRIPrep data.

In [ ]:
complete_runs = run_qc[~run_qc["incomplete"]]

for _, r in complete_runs.iterrows():
    sub = r["sub"]
    func = FMRIPREP / sub / "ses-01" / "func"
    prefix = f"{sub}_ses-01_task-{r['run_label']}"
    out_dir = TEDANA / sub / "ses-01" / "func" / prefix
    mni_file = TEDANA / sub / "ses-01" / "func" / f"{prefix}_space-MNI152NLin2009cAsym_res-2_desc-denoised_bold.nii.gz"
    if mni_file.exists():
        continue

    # tedana
    echoes = [str(func / f"{prefix}_echo-{i}_desc-preproc_bold.nii.gz") for i in range(1, 6)]
    mask = [str(m) for m in func.glob(f"{prefix}_*desc-brain_mask.nii.gz") if "space-" not in m.name][0]
    out_dir.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["tedana", "-d", *echoes, "-e", *[str(t) for t in ECHO_TIMES],
                    "--out-dir", str(out_dir), "--prefix", prefix, "--tedpca", "aic",
                    "--n-threads", "4", "--mask", mask])

    # warp the denoised data to MNI with fMRIPrep's transforms
    denoised = str(out_dir / f"{prefix}_desc-denoised_bold.nii.gz")
    reference = str(func / f"{prefix}_space-MNI152NLin2009cAsym_res-2_boldref.nii.gz")
    bold_to_t1 = str(sorted(func.glob(f"{prefix}_from-boldref_to-T1w_mode-image*_xfm.txt"))[0])
    t1_to_mni = str(sorted((FMRIPREP / sub).glob("**/anat/*from-T1w_to-MNI152NLin2009cAsym_mode-image_xfm.h5"))[0])
    subprocess.run(["apptainer", "exec", "--cleanenv", "-B", f"{ROOT}:{ROOT}", str(SIF),
                    "antsApplyTransforms", "-d", "3", "-e", "3", "-i", denoised, "-r", reference,
                    "-t", t1_to_mni, "-t", bold_to_t1, "-n", "LanczosWindowedSinc", "--float", "1",
                    "-o", str(mni_file)])
    print("done", prefix)

## 5. QC summary

In [ ]:
# tedana: components accepted / rejected per run
rows = []
for metrics_file in sorted(TEDANA.glob("sub-*/ses-01/func/*/*desc-tedana_metrics.tsv")):
    m = pd.read_csv(metrics_file, sep="\t")
    rows.append({"run": metrics_file.parent.name,
                 "components": len(m),
                 "accepted": (m["classification"] == "accepted").sum(),
                 "rejected": (m["classification"] == "rejected").sum()})
tedana_summary = pd.DataFrame(rows)
tedana_summary.to_csv(TEDANA / "tedana_summary.csv", index=False)
tedana_summary.describe().round(1)

In [ ]:
# motion by task (kept runs only)
kept = run_qc[~run_qc["run_excluded"]]
kept.groupby("task")[["n_vols", "mean_fd_raw", "mean_fd_filt", "pct_censored", "minutes_retained"]].agg(["mean", "std"]).round(2)